In [2]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np
import gzip
from collections import defaultdict

# ==============================
# File paths (CHANGE THESE)
# ==============================
tpm_file = "/content/drive/MyDrive/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
sample_attr_file = "/content/drive/MyDrive/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
output_file = "/content/drive/MyDrive/gtex_gene_tissue_expression.csv"



In [8]:
CHUNK_SIZE = 1000

# ==============================
# 1️⃣ Load sample annotation
# ==============================
sample_attr = pd.read_csv(sample_attr_file, sep="\t")
sample_to_tissue = sample_attr[['SAMPID', 'SMTSD']].set_index('SAMPID')



/tmp/ipython-input-1293461313.py:6: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  sample_attr = pd.read_csv(sample_attr_file, sep="\t")


In [6]:
# ==============================
# 2️⃣ Read header manually
# ==============================
with gzip.open(tpm_file, 'rt') as f:
    f.readline()
    f.readline()
    header = f.readline().strip().split("\t")

sample_ids = header[2:]

# Map tissue → column indices
tissue_groups = defaultdict(list)

for i, samp in enumerate(sample_ids):
    if samp in sample_to_tissue.index:
        tissue = sample_to_tissue.loc[samp, 'SMTSD']
        tissue_groups[tissue].append(i)

tissue_names = list(tissue_groups.keys())

print("Tissues found:", len(tissue_names))

Tissues found: 68


In [10]:
from tqdm import tqdm

# ==============================
# 3️⃣ Chunked reader (NO dtype forcing)
# ==============================
reader = pd.read_csv(
    tpm_file,
    sep="\t",
    skiprows=2,
    chunksize=CHUNK_SIZE
)

first_write = True

for chunk in tqdm(reader):

    gene_ids = chunk.iloc[:, 0].astype(str)
    gene_symbols = chunk.iloc[:, 1].astype(str)

    # Convert ONLY expression columns to float32
    expr = chunk.iloc[:, 2:].astype(np.float32).values

    # log2 transform
    np.log2(expr + 1, out=expr)

    # compute mean per tissue
    result = []

    for tissue in tissue_names:
        cols = tissue_groups[tissue]
        tissue_mean = expr[:, cols].mean(axis=1)
        result.append(tissue_mean)

    result = np.column_stack(result)

    out_df = pd.DataFrame(result, columns=tissue_names)
    out_df.insert(0, "GeneSymbol", gene_symbols.values)
    out_df.insert(0, "EnsemblID", gene_ids.values)

    out_df.to_csv(
        output_file,
        mode='w' if first_write else 'a',
        header=first_write,
        index=False
    )

    first_write = False
    print("Processed chunk")

print("Finished successfully.")


1it [00:09,  9.31s/it]

Processed chunk


2it [00:19, 10.00s/it]

Processed chunk


3it [00:30, 10.22s/it]

...

Processed chunk


74it [12:59, 10.65s/it]

Processed chunk


75it [13:05, 10.47s/it]

Processed chunk
Finished successfully.


In [11]:
import pandas as pd
data = pd.read_csv(output_file)
print(data.head())
data.info()

           EnsemblID   GeneSymbol  Whole Blood  Brain - Frontal Cortex (BA9)  \
0  ENSG00000290825.2     DDX11L16     0.028955                      0.018874   
1  ENSG00000223972.6      DDX11L1     0.003192                      0.000000   
2  ENSG00000310526.1       WASH7P     1.731737                      0.944307   
3  ENSG00000243485.6  MIR1302-2HG     0.011722                      0.014546   
4  ENSG00000237613.3      FAM138A     0.013435                      0.026206   

   Brain - Cerebellar Hemisphere  Brain - Substantia nigra  \
0                       0.025773                  0.017392   
1                       0.000000                  0.000000   
2                       2.216555                  0.795484   
3                       0.023820                  0.012514   
4                       0.021936                  0.024687   

   Brain - Anterior cingulate cortex (BA24)  Brain - Amygdala  \
0                                  0.016015          0.018953   
1               